# 00B · 传感器、坐标系与时间：自动驾驶模型的输入为什么容易错？

深度学习模型通常把输入写成 tensor，但自动驾驶的 tensor 有三个经常被忽略的语义：**它来自哪个传感器、在哪个坐标系、对应哪个时间**。这三件事错一个，模型可能仍然能运行，却学到错误的几何关系。

本节只做领域引入。你不需要先掌握完整的 SE(3) 群论；先理解 frame、pose、extrinsic、ego-motion、timestamp offset 和 interpolation，下一篇 `01` 再实现矩阵细节。

| 名词 | 本教程中的工作定义 |
|---|---|
| ego frame | 以自车为原点、约定前/左/上方向的坐标系 |
| sensor frame | 相机、LiDAR、radar 各自的测量坐标系 |
| world/map frame | 相对稳定的地图或世界参考系 |
| extrinsic | sensor frame 与 ego frame 之间的安装变换 |
| ego-motion | 自车在时间上的位姿变化 |
| timestamp offset | 不同传感器观测的真实时间不一致 |


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def yaw_transform(yaw_rad, translation):
    c, s = np.cos(yaw_rad), np.sin(yaw_rad)
    transform = np.array([[c, -s, translation[0]], [s, c, translation[1]], [0, 0, 1.0]])
    return transform

def apply_transform(points_xy, transform):
    homogeneous = np.c_[points_xy, np.ones(len(points_xy))]
    return (transform @ homogeneous.T).T[:, :2]

world_points = np.array([[12, 0], [15, 1], [18, -1], [22, 0.5]])
ego_pose_t0 = yaw_transform(np.deg2rad(0), [0, 0])
ego_pose_t1 = yaw_transform(np.deg2rad(12), [2.0, 0.5])
points_in_ego_t0 = apply_transform(world_points, np.linalg.inv(ego_pose_t0))
points_in_ego_t1 = apply_transform(world_points, np.linalg.inv(ego_pose_t1))

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(world_points[:, 0], world_points[:, 1], label="world/map")
ax.scatter(points_in_ego_t0[:, 0], points_in_ego_t0[:, 1], label="ego at t0")
ax.scatter(points_in_ego_t1[:, 0], points_in_ego_t1[:, 1], label="ego at t1")
ax.set_aspect("equal")
ax.set(xlabel="x / m", ylabel="y / m", title="The same world actors have different ego-frame coordinates")
ax.legend()


In [ ]:
# 时间错位：一个横穿道路的 actor 在不同 timestamp 被投影到不同位置。
def actor_position(t, speed=2.5, start=-4.0):
    return np.array([12.0, start + speed * t])

camera_time = 1.00
lidar_time = 1.12
camera_actor = actor_position(camera_time)
lidar_actor = actor_position(lidar_time)
print("camera timestamp:", camera_time, "actor:", camera_actor)
print("lidar timestamp: ", lidar_time, "actor:", lidar_actor)
print("naive fusion error / m:", np.linalg.norm(camera_actor - lidar_actor))
print("domain lesson: synchronization is a model input contract, not a preprocessing footnote")


In [ ]:
from ipywidgets import FloatSlider, interact

def show_alignment(timestamp_offset_ms=0.0, yaw_error_deg=0.0):
    transform = yaw_transform(np.deg2rad(12 + yaw_error_deg), [2.0, 0.5])
    aligned = apply_transform(world_points, np.linalg.inv(transform))
    stale = actor_position(1.0 + timestamp_offset_ms / 1000.0)
    print(f"timestamp offset: {timestamp_offset_ms:.0f} ms")
    print(f"calibration yaw error: {yaw_error_deg:.1f} deg")
    print(f"one moving actor's stale-position shift: {np.linalg.norm(stale - actor_position(1.0)):.3f} m")
    print("next deep dive: 01_se3_calibration_projection.ipynb")

interact(
    show_alignment,
    timestamp_offset_ms=FloatSlider(min=-200, max=200, step=10, value=0, description="offset / ms"),
    yaw_error_deg=FloatSlider(min=-8, max=8, step=0.5, value=0, description="yaw / deg"),
)


## 领域检查点

- 为什么把所有传感器直接 concatenate 成一个 tensor 不能解决坐标不一致？
- ego-motion compensation 解决的是“传感器在动”，还是“目标在动”？两者如何区分？
- 如果相机延迟 100 ms，车辆速度 15 m/s，静态目标在 ego frame 中会产生多大的位置偏差？

**下一步**：打开 `01_se3_calibration_projection.ipynb`，把这里的 2D toy transform 换成 3D 齐次变换、相机投影和标定误差分析。
